# module-base-class-custom — ex1: build Module: __setattr__ registers params, parameters() walks recursively

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `module-base-class-custom`. Running the final beacon cell reports progress against the `Backprop: Module base class custom` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Module base class custom` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`module-base-class-custom`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "module-base-class-custom"
DD_SUBTOPIC = "Backprop: Module base class custom"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `Module` base class from scratch — quick refresher

A minimal `nn.Module` clone needs THREE things:

1. **`__init__`** — initialize an internal store for parameters / submodules. Don't rely on the subclass to call `super().__init__()` and *still* work if it doesn't — but the canonical pattern is to require it.
2. **`__setattr__`** — intercept attribute assignment. When the value is a `Parameter` (or `Module`), register it in the parameters / submodules store. Plain attributes (ints, lists, etc.) bypass.
3. **`parameters()` walker** — yields every `Parameter`, walking recursively into submodules via depth-first traversal.

```python
class Module:
    def __init__(self):
        self._parameters = {}
        self._modules = {}
    def __setattr__(self, name, value):
        if isinstance(value, Parameter):
            self._parameters[name] = value
        elif isinstance(value, Module):
            self._modules[name] = value
        object.__setattr__(self, name, value)
    def parameters(self):
        yield from self._parameters.values()
        for m in self._modules.values():
            yield from m.parameters()
    def forward(self, *args, **kwargs):
        raise NotImplementedError
```

**Why `__setattr__`.** Auto-registration is what makes `self.weight = Parameter(...)` Just Work. Without it the user has to call `self.register_parameter('weight', w)` explicitly.

### Exercise 1 — build Module: __setattr__ registers params, parameters() walks recursively

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the `__setattr__`-as-registrar pattern to build a minimal `Module` base class whose `parameters()` walks all directly-assigned Parameters plus those of any submodule, transitively.
> Keywords: module, setattr, parameters, recursive-walker, submodule
> ```

**KCs targeted:** `module-base-class-custom`, `parameter-subclass-of-tensor`

Implement `Module` and a sample `Parameter` subclass so the test cell can build a tiny 2-layer model and verify all parameters are discoverable.

**1. `Parameter(MiniTensor)`** — already covered in batch-3 / batch-4. Subclass `MiniTensor`, default `requires_grad=True`. Just re-define it here so the drill is self-contained.

**2. `Module` base class.** Required surface:
   - `__init__(self)` — initialize `self._parameters = {}` and `self._modules = {}` BEFORE any other attribute is set. Use `object.__setattr__` for this bootstrap so the custom `__setattr__` doesn't recurse on the registry dicts themselves.
   - `__setattr__(self, name, value)` — when `value` is a `Parameter`, register it in `self._parameters[name]`. When `value` is a `Module`, register in `self._modules[name]`. Then ALWAYS call `object.__setattr__(self, name, value)` so the attribute is also accessible via normal `.name` lookup.
   - `parameters(self)` — generator that yields:
     1. Every direct `Parameter` (`self._parameters.values()`).
     2. Every parameter of every submodule (`m.parameters()` for `m` in `self._modules.values()`).
   - `forward(self, *args, **kwargs)` — abstract; raise `NotImplementedError()`.

**Why `__setattr__` not `register_parameter`.** The intercept lets users write `self.weight = Parameter(...)` and have it Just Work. Without it, every layer would need explicit `register_parameter` calls.

**Bootstrap order matters.** `object.__setattr__(self, '_parameters', {})` in `__init__` is crucial. If you wrote `self._parameters = {}`, your custom `__setattr__` would fire and try to read `self._parameters` to register the value — but `self._parameters` doesn't exist yet. Stack overflow.

In [ ]:
class Parameter(MiniTensor):
    def __init__(self, array, requires_grad: bool = True):
        super().__init__(array, requires_grad=requires_grad)


class Module:
    def __init__(self):
        # bootstrap: bypass our own __setattr__ to install the registries
        object.__setattr__(self, '_parameters', {})
        object.__setattr__(self, '_modules', {})

    def __setattr__(self, name, value):
        if isinstance(value, Parameter):
            self._parameters[name] = value
            # remove any prior submodule slot with the same name
            self._modules.pop(name, None)
        elif isinstance(value, Module):
            self._modules[name] = value
            self._parameters.pop(name, None)
        else:
            self._parameters.pop(name, None)
            self._modules.pop(name, None)
        # always also expose the attr through normal lookup
        object.__setattr__(self, name, value)

    def parameters(self):
        for p in self._parameters.values():
            yield p
        for m in self._modules.values():
            yield from m.parameters()

    def forward(self, *args, **kwargs):
        raise NotImplementedError()


<details><summary>Solution</summary>

```python
class Parameter(MiniTensor):
    def __init__(self, array, requires_grad: bool = True):
        super().__init__(array, requires_grad=requires_grad)


class Module:
    def __init__(self):
        # bootstrap: bypass our own __setattr__ to install the registries
        object.__setattr__(self, '_parameters', {})
        object.__setattr__(self, '_modules', {})

    def __setattr__(self, name, value):
        if isinstance(value, Parameter):
            self._parameters[name] = value
            # remove any prior submodule slot with the same name
            self._modules.pop(name, None)
        elif isinstance(value, Module):
            self._modules[name] = value
            self._parameters.pop(name, None)
        else:
            self._parameters.pop(name, None)
            self._modules.pop(name, None)
        # always also expose the attr through normal lookup
        object.__setattr__(self, name, value)

    def parameters(self):
        for p in self._parameters.values():
            yield p
        for m in self._modules.values():
            yield from m.parameters()

    def forward(self, *args, **kwargs):
        raise NotImplementedError()
```

**Why `object.__setattr__` in `__init__`.** During `__init__`, the registries don't exist yet. If you write `self._parameters = {}` the custom `__setattr__` fires and tries to inspect `self._parameters` — infinite recursion (or AttributeError, depending on order). `object.__setattr__(self, '_parameters', {})` bypasses the override and installs the dict directly.

**Why both registries are updated on every assignment.** If a user writes `self.layer = SomeSubmodule()` and then later `self.layer = Parameter(...)`, the previous `_modules['layer']` must be evicted — otherwise `parameters()` would yield the old submodule's params even though `self.layer` no longer points to it. The `.pop(name, None)` calls handle this defensively.

**Recursive walk via `yield from`.** A non-recursive `parameters()` (only yielding `_parameters.values()`) would miss everything in submodules — a 2-layer MLP's `Linear` weights would be invisible. `yield from m.parameters()` makes the walker depth-first and transitively complete.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()